# Spike Thresholding — SLURM Batch Submission

Submits one chipmunk SLURM job per interval to run `spike_thresholding_worker.py`.

Worker uses `neo.io.BlackrockIO` to load the stitched NSP-2 NS5 (micro-electrode channels, 30 kHz, uV),
applies a 300-6000 Hz bandpass + 60 Hz notch, then solves for the threshold multiplier `k` that
yields ≈20 Hz average spike rate, detects negative-deflection peaks, and bins into a 1 kHz binary spike train.

Output per interval: `neural/spike_thresholding/binary_spiketrain.npy`, `spike_stats.json`, `channel_names.json`, `_SUCCESS`.

In [2]:
import json
import subprocess
from pathlib import Path
import pandas as pd

In [9]:
# ── Config ────────────────────────────────────────────────────────────────────
patient = "YFA"
write_root = Path("/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new")
patient_dir = write_root / patient
vad_data_dir = patient_dir / "vad_data"
manifest_path = patient_dir / f"{patient}_vad_merged_intervals.csv"

# SLURM
partition = "Universal"
qos = "default_tier"
cpus = 4
mem = "16G"

# Python interpreter — direct path avoids conda activation issues in batch env
python_bin = "/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3"
worker_path = Path("/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/signal_processing/spike_thresholding_worker.py")

# Spike thresholding params (passed to worker)
nsp = 2
target_hz = 20.0
notch_hz = 60.0
out_fs = 1000

logs_dir = patient_dir / "spike_slurm_logs"
batch_dir = patient_dir / "spike_slurm_scripts"
logs_dir.mkdir(parents=True, exist_ok=True)
batch_dir.mkdir(parents=True, exist_ok=True)

print("manifest:", manifest_path)
print("logs:", logs_dir)
print("scripts:", batch_dir)

manifest: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/YFA_vad_merged_intervals.csv
logs: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/spike_slurm_logs
scripts: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/spike_slurm_scripts


In [10]:
# ── Scan eligible intervals ───────────────────────────────────────────────────
manifest = pd.read_csv(manifest_path, index_col=0)
print(f"manifest rows: {len(manifest)}")

rows = []
for _, row in manifest.iterrows():
    iid = str(row["interval_id"])
    idir = vad_data_dir / iid
    ns5_path = next((idir / "neural").glob(f"*_NSP-{nsp}.ns5"), None)
    stitch_ok = (idir / "_SUCCESS").exists()
    spike_ok = (idir / "neural" / "spike_thresholding" / "_SUCCESS").exists()
    rows.append({
        "interval_id": iid,
        "toc": row["toc"],
        "duration_s": float(row["duration_s"]),
        "interval_dir": idir,
        "ns5_path": ns5_path,
        "stitch_ok": stitch_ok,
        "spike_ok": spike_ok,
    })

df = pd.DataFrame(rows)
eligible = df[df["stitch_ok"] & df["ns5_path"].notna() & ~df["spike_ok"]].copy()

print(f"total intervals: {len(df)}")
print(f"  stitch complete: {df.stitch_ok.sum()}")
print(f"  spike complete:  {df.spike_ok.sum()}")
print(f"  eligible to run: {len(eligible)}")
eligible[["interval_id", "duration_s", "toc"]].head(10)

manifest rows: 1083
total intervals: 1083
  stitch complete: 1083
  spike complete:  0
  eligible to run: 1083


,interval_id,duration_s,toc
0,20240423-124841_0000,223.561,20240423-124841
1,20240423-124841_0001,161.400,20240423-124841
2,20240423-124841_0002,8.200,20240423-124841
3,20240423-124841_0003,223.302,20240423-124841
4,20240423-124841_0004,545.107,20240423-124841
5,20240423-124841_0005,1.200,20240423-124841
6,20240423-124841_0006,4.600,20240423-124841
7,20240423-124841_0007,1128.305,20240423-124841
8,20240423-124841_0008,1450.250,20240423-124841
9,20240423-124841_0009,1313.376,20240423-124841


In [11]:
# ── Submit SLURM jobs ─────────────────────────────────────────────────────────
# Set DRY_RUN = True to generate scripts without submitting.
# Set SUBMIT_ONE = True to only submit the first job (for validation).
DRY_RUN = False
SUBMIT_ONE = False   # flip to False once validated

submitted = []
skipped = []
failed_submissions = []

for _, row in eligible.iterrows():
    iid = row["interval_id"]
    idir = row["interval_dir"]

    if (idir / "neural" / "spike_thresholding" / "_SUCCESS").exists():
        skipped.append(iid)
        continue

    job_name = f"spike_{iid}"

    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --partition={partition}
#SBATCH --qos={qos}
#SBATCH --cpus-per-task={cpus}
#SBATCH --mem={mem}
#SBATCH --output={logs_dir}/{iid}_%j.out
#SBATCH --error={logs_dir}/{iid}_%j.err

set -eo pipefail

PYTHON={python_bin}

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "INTERVAL: {iid}"

$PYTHON {worker_path} "{idir}" \\
    --nsp {nsp} \\
    --target-hz {target_hz} \\
    --notch {notch_hz} \\
    --out-fs {out_fs}

echo "END: $(date)"
"""

    sbatch_path = batch_dir / f"{iid}.sbatch"
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        print(f"dry-run: {iid}")
        submitted.append((iid, "dry-run"))
    else:
        try:
            res = subprocess.run(
                ["sbatch", str(sbatch_path)],
                capture_output=True, text=True, check=True,
            )
            submitted.append((iid, res.stdout.strip()))
            print(f"submitted: {iid} -> {res.stdout.strip()}")
        except subprocess.CalledProcessError as e:
            failed_submissions.append((iid, e.stderr.strip()))
            print(f"FAILED: {iid}\n{e.stderr}")

    if SUBMIT_ONE and not DRY_RUN and submitted:
        print("SUBMIT_ONE=True — stopping after first submission.")
        break

print(f"\nsubmitted: {len(submitted)}  skipped: {len(skipped)}  failed: {len(failed_submissions)}")

submitted: 20240423-124841_0000 -> Submitted batch job 218245
submitted: 20240423-124841_0001 -> Submitted batch job 218246
submitted: 20240423-124841_0002 -> Submitted batch job 218247
submitted: 20240423-124841_0003 -> Submitted batch job 218248
submitted: 20240423-124841_0004 -> Submitted batch job 218249
submitted: 20240423-124841_0005 -> Submitted batch job 218250
submitted: 20240423-124841_0006 -> Submitted batch job 218251
submitted: 20240423-124841_0007 -> Submitted batch job 218252
submitted: 20240423-124841_0008 -> Submitted batch job 218253
submitted: 20240423-124841_0009 -> Submitted batch job 218254
submitted: 20240423-124841_0010 -> Submitted batch job 218255
submitted: 20240423-124841_0011 -> Submitted batch job 218256
submitted: 20240423-124841_0012 -> Submitted batch job 218257
submitted: 20240423-171352_0000 -> Submitted batch job 218258
submitted: 20240423-171352_0001 -> Submitted batch job 218259
submitted: 20240423-171352_0002 -> Submitted batch job 218260
submitte

In [ ]:
# ── Check first submission ─────────────────────────────────────────────────────
# After running the test job, check its output here.
if submitted:
    first_iid, job_info = submitted[0]
    out_path = vad_data_dir / first_iid / "neural" / "spike_thresholding"
    print(f"interval: {first_iid}")
    print(f"job:      {job_info}")
    print(f"output:   {out_path}")
    print()

    success = (out_path / "_SUCCESS").exists()
    error = (out_path / "error.txt").exists()
    print(f"_SUCCESS: {success}")
    print(f"error.txt: {error}")

    if error:
        print("\nError contents:")
        print((out_path / "error.txt").read_text())

    if success:
        import numpy as np
        spike_bin = np.load(str(out_path / "binary_spiketrain.npy"))
        with open(out_path / "spike_stats.json") as f:
            stats = json.load(f)
        ch_names = json.loads((out_path / "channel_names.json").read_text())
        print(f"\nbinary_spiketrain shape: {spike_bin.shape}  dtype: {spike_bin.dtype}")
        print(f"n_channels: {stats['n_channels']}  duration: {stats['duration_s']:.1f}s")
        print(f"achieved avg rate: {stats['achieved_avg_rate_Hz']:.2f} Hz  k={stats['k']:.3f}")
        print(f"per-channel rate range: {min(stats['per_channel_rates_Hz']):.1f} – {max(stats['per_channel_rates_Hz']):.1f} Hz")
        print(f"first 5 channel names: {ch_names[:5]}")

In [ ]:
# ── Summary of completed intervals ────────────────────────────────────────────
# Re-scan after jobs finish to get completion stats.
import numpy as np

done = []
for iid in df["interval_id"]:
    p = vad_data_dir / iid / "neural" / "spike_thresholding"
    if (p / "_SUCCESS").exists():
        stats_path = p / "spike_stats.json"
        if stats_path.exists():
            with open(stats_path) as f:
                stats = json.load(f)
            done.append({
                "interval_id": iid,
                "n_channels": stats["n_channels"],
                "duration_s": stats["duration_s"],
                "achieved_hz": stats["achieved_avg_rate_Hz"],
                "k": stats["k"],
            })

summary = pd.DataFrame(done)
print(f"Completed: {len(summary)} / {len(df)}")
if not summary.empty:
    display(summary.describe())
    display(summary.head(10))